# Stage 2 — Bias Correction & Negative Sampling

Two methodologically critical steps:

1. **Per-species spatial thinning** — cap each species at 1 record per 0.01° cell to reduce spatial autocorrelation bias
2. **Target-group background sampling** — draw pseudo-absences from the full iNat observation pool (all quality grades) to correct for observer effort bias

**Correctness check (Cell 9):** spatial map of presences vs backgrounds for 3 representative species.  
If backgrounds are not broadly spread across the study area, stop and flag before Stage 3.

In [ ]:
import sys
from pathlib import Path
_here = Path.cwd()
_root = _here.parent if _here.name == 'notebooks' else _here
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)-8s %(name)s  %(message)s', force=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from pipeline.sampling import (
    spatial_thin, build_background_pool, sample_backgrounds,
    check_background_coverage, run_sampling,
)
from pipeline.ingest import load_heat_map, build_sensor_climatology
from config import (
    ALIGNED_PATH, SAMPLED_PATH, INAT_PATH, HEAT_MAP_PATH,
    GRID_CELL_DEG, MIN_PRESENCE_COUNT,
)

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 110})
print('Setup complete')

## 1. Load Stage 1 presences

In [ ]:
presences = pd.read_parquet(ALIGNED_PATH)
print(f'Presence records : {len(presences):,}')
print(f'Species          : {presences["taxon_name"].nunique()}')
print(f'Columns          : {list(presences.columns)}')
display(presences.head(3))

In [ ]:
counts = presences['taxon_name'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

counts.head(30).plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='none')
axes[0].set_title('Top 30 species by record count (pre-thinning)')
axes[0].set_xlabel('Species'); axes[0].set_ylabel('Records')
axes[0].tick_params(axis='x', rotation=60, labelsize=7)

counts.plot(kind='hist', bins=40, ax=axes[1], color='steelblue', edgecolor='none')
axes[1].axvline(MIN_PRESENCE_COUNT, color='red', linestyle='--',
                label=f'Min threshold ({MIN_PRESENCE_COUNT})')
axes[1].set_title('Distribution of records per species')
axes[1].set_xlabel('Records'); axes[1].legend()
plt.tight_layout(); plt.show()

## 2. Per-species spatial thinning
Retain at most 1 record per 0.01° (≈1.1 km) grid cell, independently per species.

In [ ]:
thinned, thin_log = spatial_thin(presences, grid_deg=GRID_CELL_DEG)

print(f'Before thinning : {len(presences):,} records, {presences["taxon_name"].nunique()} species')
print(f'After thinning  : {len(thinned):,} records, {thinned["taxon_name"].nunique()} species')
print(f'Records removed : {len(presences)-len(thinned):,}  ({len(thin_log)} species affected)')

In [ ]:
# Before vs after counts per species
before = presences['taxon_name'].value_counts().rename('before')
after  = thinned['taxon_name'].value_counts().rename('after')
comparison = pd.concat([before, after], axis=1).sort_values('before', ascending=False)

fig, ax = plt.subplots(figsize=(14, 4))
x = range(len(comparison))
ax.bar(x, comparison['before'], label='Before thinning', color='#90CAF9', width=0.8)
ax.bar(x, comparison['after'],  label='After thinning',  color='#1565C0', width=0.8)
ax.axhline(MIN_PRESENCE_COUNT, color='red', linestyle='--', label=f'Min ({MIN_PRESENCE_COUNT})')
ax.set_title('Records per species before vs after spatial thinning')
ax.set_xlabel('Species (sorted by pre-thinning count)')
ax.set_ylabel('Records')
ax.set_xticks([]); ax.legend()
plt.tight_layout(); plt.show()

# Check for species that fall below threshold after thinning
below = comparison[comparison['after'] < MIN_PRESENCE_COUNT]
print(f'Species below threshold after thinning: {len(below)}')
if len(below):
    print(below.to_string())

## 3. Build target-group background pool
Load ALL iNat observations (any quality grade) and spatially match to sensor climatology.

In [ ]:
sensor_df   = load_heat_map(HEAT_MAP_PATH)
climatology = build_sensor_climatology(sensor_df)
bg_pool     = build_background_pool(INAT_PATH, climatology)

print(f'Background pool size : {len(bg_pool):,}')
print(f'Temp range           : {bg_pool.temperature_c.min():.1f} – {bg_pool.temperature_c.max():.1f} °C')
print(f'Match distance (m)   : mean={bg_pool._matched_dist_m.mean():.0f}, max={bg_pool._matched_dist_m.max():.0f}')
display(bg_pool.head(3))

In [ ]:
# Spatial density of background pool — this should reflect observer effort
fig, ax = plt.subplots(figsize=(8, 7))
h = ax.hexbin(bg_pool['lon'], bg_pool['lat'], gridsize=40, cmap='YlOrRd', mincnt=1)
plt.colorbar(h, ax=ax, label='Observation count')
ax.set_title('Background pool spatial density\n(should reflect observer effort, not uniform)')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
plt.tight_layout(); plt.show()

## 4. Sample backgrounds

In [ ]:
N_MULTIPLIER = 10  # backgrounds = 10x total thinned presences
n_bg = int(len(thinned) * N_MULTIPLIER)
print(f'Requesting {n_bg:,} background samples ({N_MULTIPLIER}× {len(thinned):,} thinned presences)')

backgrounds = sample_backgrounds(bg_pool, n_samples=n_bg)
ratio = len(backgrounds) / len(thinned)
print(f'Actual background:presence ratio: {ratio:.1f}x')

## 5. ✅ Correctness check — background vs presence spatial distribution

For each of 3 representative species, the background points (grey) should broadly overlap with the presence points (colour).  
If backgrounds are absent from regions where a species occurs, the model has no negative signal there — **stop and flag.**

In [ ]:
# Pick 3 representative species: high, median, and low record count (post-thinning)
counts_thinned = thinned['taxon_name'].value_counts()
rep_species = [
    counts_thinned.index[0],
    counts_thinned.index[len(counts_thinned)//2],
    counts_thinned.index[-1],
]
print('Representative species:', rep_species)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, sp in zip(axes, rep_species):
    sp_df = thinned[thinned['taxon_name'] == sp]
    ax.scatter(backgrounds['lon'], backgrounds['lat'],
               s=1, alpha=0.15, c='grey', label='Background')
    ax.scatter(sp_df['lon'], sp_df['lat'],
               s=20, alpha=0.8, c='crimson', zorder=5,
               label=f'Presence (n={len(sp_df)})')
    ax.set_title(sp, fontsize=9, wrap=True)
    ax.set_xlabel('Lon'); ax.set_ylabel('Lat')
    ax.legend(fontsize=8, markerscale=2)

plt.suptitle('Presence (red) vs Background (grey) — backgrounds should broadly overlap study area',
             fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Programmatic coverage check
check_background_coverage(thinned, backgrounds)

## 6. Feature distribution comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for col, ax in zip(['temperature_c', 'humidity_pct', 'day_of_year'], axes):
    ax.hist(thinned[col], bins=40, alpha=0.6, density=True,
            color='crimson', label='Presences')
    ax.hist(backgrounds[col], bins=40, alpha=0.5, density=True,
            color='grey', label='Backgrounds')
    ax.set_title(col.replace('_', ' ').title())
    ax.legend()
plt.suptitle('Feature distributions: presences vs backgrounds', y=1.02)
plt.tight_layout()
plt.show()
print('If presences and backgrounds have very similar distributions, the model')
print('will have low discriminatory power — but this may reflect genuine habitat')
print('ubiquity for some species. Per-species AUC in Stage 4 will confirm.')

## 7. Run full pipeline & save

In [ ]:
sampled = run_sampling(n_background_multiplier=10.0)

## 8. Verify saved output

In [ ]:
if SAMPLED_PATH.exists():
    s = pd.read_parquet(SAMPLED_PATH)
    print(f'sampled.parquet  shape    : {s.shape}')
    print(f'Presence=1 rows           : {(s.presence==1).sum():,}')
    print(f'Presence=0 (background)   : {(s.presence==0).sum():,}')
    print(f'Species in presence rows  : {s[s.presence==1]["taxon_name"].nunique()}')
    print(f'Null taxon_name (bg rows) : {s["taxon_name"].isna().sum():,}')
    print(f'Columns: {list(s.columns)}')
    print()
    print('Null counts:')
    print(s.isnull().sum())
    display(s.head(5))
else:
    print('sampled.parquet not found')